# MAVE Data Analysis I: Quantification Tutorial Answers

This document provides answers and explanations for all exercises in the quantification tutorial.

---

## Exercise 1: Checking Adapter Trimming Efficiency

### Question 1.1: How many reads were processed by cutadapt?

**Answer:** 500,000 reads

**Explanation:** This is shown in the cutadapt log:
```
Total reads processed:                 500,000
```

This represents the number of reads in the raw FASTQ file that was downsampled for the tutorial.

---

### Question 1.2: What percentage of reads contained the Illumina adapter sequence?

**Answer:** 97.0%

**Calculation:**
```
(485,153 / 500,000) × 100 = 97.0%
```

**Explanation:** From the cutadapt log:
```
Reads with adapters:                   485,153 (97.0%)
```

This high percentage is expected because:
* The insert size (~280bp) is shorter than the read length (450bp)  
* After sequencing through the insert, the sequencer reads into the Illumina adapter  
* Most reads capture the adapter, which is why adapter trimming is essential

**Consistency across samples:** Yes, adapter detection rates should be very consistent across samples (typically 95-98% for this dataset), as all samples have similar insert sizes and were sequenced under identical conditions.

---

## Exercise 2: Checking Primer Trimming Success

### Question 2.1: What percentage of reads contained both the forward and reverse primer?

**Answer:** 99.7%

**Calculation:**
```
(498,573 / 500,000) × 100 = 99.7%
```

**Explanation:** From the cutadapt log:
```
Reads with adapters:                   498,573 (99.7%)
```

Note: cutadapt calls linked primers "adapters" in its output. This very high percentage (>99%) indicates excellent library quality and primer detection.

---

### Question 2.2: What reason might there be for some reads to not be trimmed?

**Possible reasons:**

1. **Sequencing errors in primer regions** - If the primer sequence contains errors, cutadapt won't recognize it
2. **Very short reads** - Reads shorter than expected may not contain both primers
3. **Degraded DNA** - Partial fragments missing one or both primers
4. **Off-target sequences** - Contaminating DNA that isn't from the library
5. **PCR artifacts** - Chimeric sequences or mispriming events

**For this dataset:** Only 0.3% of reads (1,427 reads) were untrimmed, which indicates excellent library quality. This small fraction likely represents sequencing errors or minor artifacts.

**Consistency across samples:** Yes, primer trimming rates should be consistent (typically >99% for high-quality SGE libraries). Large variations might indicate sample-specific problems.

---

## Exercise 3: MultiQC Quality Overview

### Question 3.1: How long were our raw reads?

**Answer:** 450 bp (base pairs)

**Where to find:** MultiQC report → FastQC (raw) section → "Sequence Length Distribution" plot

**Explanation:** This is the standard read length for this Illumina sequencing run. All samples should show the same length since they were sequenced together.

---

### Question 3.2: At what position (bp) does adapter content start to appear?

**Answer:** Around position 280-300 bp

**Where to find:** MultiQC report → FastQC (raw) section → "Adapter Content" plot

**Explanation:** Adapter content sharply increases at ~280-300bp because:
- The DNA insert is approximately 280bp long
- After sequencing through the insert, the sequencer begins reading the Illumina adapter
- The steep rise indicates most fragments are a similar size

---

### Question 3.3: What percentage of reads contain adapter sequences by the end of the read?

**Answer:** Approximately 95-100%

**Where to find:** FastQC (raw) → "Adapter Content" plot (look at position 450bp)

**Explanation:** By the end of the 450bp read, nearly all sequences show adapter content because the insert is only ~280bp, leaving ~170bp of adapter sequence to be read.

---

### Question 3.4: What percentage of reads contain adapter sequences after adapter trimming has taken place?

**Answer:** <0.5% (essentially 0%)

**Where to find:** MultiQC report → FastQC (adapter trimmed) section → "Adapter Content" plot

**Explanation:** The adapter content drops to nearly zero after cutadapt removes the adapters, confirming successful trimming. The tiny residual amount (<0.5%) likely represents reads where the adapter wasn't perfectly detected.

---

### Question 3.5: What is the average read length after adapter trimming and after primer trimming? Why are the reads not all the same length after trimming?

**Answer:**

**After adapter trimming:** ~295-305 bp (still variable)

**After primer trimming:** ~240-250 bp (still variable)

**Where to find:** FastQC sections for each trimming stage → "Sequence Length Distribution"

**Why reads vary in length:**

1. **Biological variation** - The variant region itself has different lengths:
   - SNVs don't change length
   - Insertions make sequences longer
   - Deletions make sequences shorter

2. **Incomplete primer matching** - Not all primers are detected at exactly the same position:
   - Sequencing errors can shift the match position
   - Cutadapt's matching algorithm allows some flexibility

3. **Variable insert sizes** - Small variations in the original DNA fragment length

**This is normal and expected** for SGE libraries containing different variant types (SNVs, indels, etc.).

---

## Exercise 4: Understanding pyQUEST Count Files and Statistics

### Part A: Interpreting pyQUEST Statistics

### Question 4.1: What percentage of input reads successfully mapped to library variants?

**Answer:** 50.7%

**Calculation:**
```
(252,984 / 498,573) × 100 = 50.7%
```

**Is this acceptable?** Yes! For SGE experiments, mapping rates of 40-60% are typical and acceptable because:
- Sequencing errors prevent some reads from matching perfectly
- Some reads are from library synthesis artifacts
- Not all reads will be perfect after trimming

**Important:** The key metric is consistency across samples, not achieving 100% mapping.

---

### Question 4.2: How well is the library represented at Day 4?

**Answer:** Excellent representation

**Evidence:**
- `zero_count_templates`: 0 (no variants completely missing)
- `low_count_templates_lt_15`: 3 (only 3 variants with <15 reads)
- `low_count_templates_lt_30`: 8 (only 8 variants with <30 reads)

**What this means:**
- All 1,278 library variants were detected
- 99.8% of variants have ≥15 reads (good coverage)
- 99.4% of variants have ≥30 reads (very good coverage)

**Interpretation:** At the baseline timepoint (Day 4), the library is well-represented with excellent coverage across nearly all variants. This is essential for detecting selection in later timepoints.

---

### Question 4.3: What does the mean vs. median tell you about the distribution?

**Given:**
- Mean: 243.25 reads per variant
- Median: 174.0 reads per variant

**Answer:** The distribution is **right-skewed** (positively skewed)

**What this means:**
- Mean > Median indicates some variants have much higher counts than others
- A few highly abundant variants are "pulling" the mean upward
- Most variants cluster around the median (174 reads)
- There's a "long tail" of high-count variants

**Is this normal?** Yes! For biological libraries, some skewness is expected as some variants may already be slightly enriched even at Day 4

---

### Question 4.4: How has library coverage changed by Day 21?

**Answer:**

| Metric | Day 4 | Day 21 | Change |
|:-------|:------|:-------|:-------|
| `zero_count_templates` | 0 | 0 | No change |
| `low_count_templates_lt_15` | 3 | 14 | +11 variants |
| `low_count_templates_lt_30` | 8 | 72 | +64 variants |
| `gini_coefficient` | 0.39 | 0.60 | +0.21 (more inequality) |

**What biological process explains these changes?**

**Negative selection!**

- **Increase in low-count variants:** Disruptive BAP1 variants cause cell death, so their counts drop dramatically (from moderate → low)
- **Increased Gini coefficient:** Coverage becomes more unequal as functional variants remain abundant while disruptive ones deplete
- **Zero-count remains 0:** Even disruptive variants still have a few reads due to sufficient sequencing depth

**Key insight:** By Day 21, selection has created a population dominated by functional variants, while disruptive variants have become rare. This is exactly what we expect for an essential gene like BAP1.

---

### Question 4.5: How has the mapping rate changed? Why?

**Answer:**

| Metric | Day 4 | Day 21 |
|:-------|:------|:-------|
| `input_reads` | 498,573 | 498,919 |
| `mapped_to_template_reads` | 252,984 | 337,858 |
| Percentage reads mapped | 50.7% | 67.7% |

**Change:** Mapping rate increased by 17 percentage points (50.7% → 67.7%)

**Why does this happen during negative selection?**

1. **Enrichment of well-defined variants** - Functional variants with clear, unambiguous sequences become more abundant

2. **Depletion of problematic sequences** - Disruptive variants that may have had sequencing errors or ambiguous sequences are removed from the population

3. **Library "purification"** - As selection progresses, the population becomes dominated by a subset of highly reproducible, well-sequenced variants

4. **Reduced complexity** - Fewer unique variants means less chance of sequencing errors creating unmappable reads

**This is a quality indicator:** Rising mapping rates over time suggest strong selection and good data quality.

---

### Part B: Exploring Query Counts

### Question 4.6: What are the two most abundant read sequences in Day 4 Rep1?

**Command:**
```bash
zcat results_5a/pyquest/5_a_Day4_Rep1.query_counts.tsv.gz | tail -n +2 | sort -k3 -n -r | head -2
```

**Answer:**

**1st most abundant:**
- Sequence: `AATGATACGGCGACCACCGATTGGGGCTTGCAGTGAGGGGTGCTGTGTATGGGTGACTATTCTTGGTTTCACAGCTGATACCCAACTCTTGTGCAACTCATGCTTTGCTAAGCGTGCTCCTGAACTGCAGCAGCGTGGACCTGGGACCCACCCTGAGTCGCATGAAGGACTTCACCAAGGGTTTCAGCCCTGAGGTAGGCTGCAGTGCCTTCATCCTGGCTCACAGCCAACTGGGCAGATCTGACCCTGAGGGCCACTGGGAATGTCGTATGCCGTCTTCTGCTTG`
- Count: 56,705
- This is the **PAM-protected sequence** (synonymous edits)

**2nd most abundant:**
- Sequence: `AATGATACGGCGACCACCGATTGGGGCTTGCAGTGAGGGGTGCTGTGTATGGGTGACTATTCTTGGTTTCACAGCTGATACCCAACTCTTGTGCAACTCATGCCTTGCTCCTGAACTGCAGCAGCGTGGACCTGGGACCCACCCTGAGTCGCATGAAGGACTTCACCAAGGGTTTCAGCCCTGAGGTAGGCTGCAGTGCCTTCATCCTGGCTCACAGCCAACTGGGCAGATCTGACCCTGAGGGCCACTGGGAATGTCGTATGCCGTCTTCTGCTTG`
- Count: 14,017

**Interpretation:** The PAM-protected variant is 4× more abundant than the second most abundant sequence, even at Day 4.

---

## Exercise 5: Exploring Library Counts

### Question 5.1: What version of pyQUEST was used to generate these counts?

**Answer:** Version 1.1.0

**Command:**
```bash
zcat results_5a/pyquest/5_a_Day4_Rep1.lib_counts.tsv.gz | head -2
```

**Output:**
```
##Command: pyquest --cpus 4 -l valiant_library_5a.pyquest.tsv -s 5_a_Day4_Rep1 -o 5_a_Day4_Rep1 5_a_Day4_Rep1.modified.fq.gz
##Version: 1.1.0
```

**Why is tool version important?**

1. **Reproducibility** - Other researchers can use the exact same version
2. **Bug tracking** - If results seem odd, you can check if that version had known issues
3. **Method comparison** - Different versions may have algorithm changes
4. **Documentation** - Essential for Methods sections in publications
5. **Troubleshooting** - Version-specific problems can be identified

**Best practice:** Always record tool versions in your analysis logs!

---

### Question 5.2: What is the name and length of the first oligo in the count table and how abundant was it? Is it a unique sequence?

**Answer:**

**Name:** `ENST00000460680.6.ENSG00000163930.10_chr3:52407928_52407929_2del0_rc`

**Length:** 284 bp

**Count:** 135 reads

**Unique:** Yes (UNIQUE = 1)

**Breaking down the name:**
- `ENST00000460680.6` - Ensembl transcript ID
- `ENSG00000163930.10` - Ensembl gene ID (BAP1)
- `chr3:52407928_52407929` - Genomic coordinates
- `2del0` - Variant type (2bp deletion at position 0)
- `rc` - Reverse complement (library is on minus strand)

**Interpretation:** This is a 2bp deletion variant with moderate abundance (135 reads) at Day 4. The unique flag (1) means this exact sequence appears only once in the library design.

---

### Question 5.3: What can you infer from the counts for the variant ENST00000460680.6.ENSG00000163930.10_chr3:52408050_1del_rc?

**Answer:** This variant shows a **decreasing trend** over time as counts drop steadily

**Interpretation:**

- This 1bp deletion is **disruptive** to BAP1 function
- Cells carrying this variant cannot survive
- The variant becomes progressively rarer as those cells die
- This is likely a **loss-of-function (LOF)** variant

**Why this matters:** In clinical genetics, decreasing variants would be classified as likely pathogenic, while stable/increasing variants might be benign.

**This is the power of SGE:** We can experimentally determine variant effects rather than relying on computational predictions! But you need to do this with normalisation and scores...the later practical for it to be reliable and give a measure of significance!!

---

## Additional Information: PAM and REF Control Sequences

### PAM and REF Counts Across Timepoints

| Sample | REF Count | PAM Count | Total Reads | % REF | % PAM |
|:-------|:----------|:----------|:------------|:------|:------|
| 5_a_Day4_Rep1 | 14,017 | 56,705 | 498,573 | 2.8% | 11.4% |
| 5_a_Day4_Rep2 | 13,895 | 56,321 | 498,499 | 2.8% | 11.3% |
| 5_a_Day4_Rep3 | 13,354 | 54,640 | 498,665 | 2.7% | 11.0% |
| 5_a_Day7_Rep1 | 8,046 | 63,183 | 498,798 | 1.6% | 12.7% |
| 5_a_Day7_Rep2 | 7,183 | 63,445 | 498,711 | 1.4% | 12.7% |
| 5_a_Day7_Rep3 | 11,087 | 62,334 | 498,655 | 2.2% | 12.5% |
| 5_a_Day10_Rep1 | 5,183 | 74,426 | 498,732 | 1.0% | 14.9% |
| 5_a_Day10_Rep2 | 3,821 | 75,254 | 498,821 | 0.8% | 15.1% |
| 5_a_Day10_Rep3 | 7,800 | 73,752 | 498,735 | 1.6% | 14.8% |
| 5_a_Day14_Rep1 | 4,286 | 97,191 | 498,815 | 0.9% | 19.5% |
| 5_a_Day14_Rep2 | 2,882 | 95,285 | 498,815 | 0.6% | 19.1% |
| 5_a_Day14_Rep3 | 8,061 | 94,641 | 498,832 | 1.6% | 19.0% |
| 5_a_Day21_Rep1 | 1,817 | 114,261 | 498,919 | 0.4% | 22.9% |
| 5_a_Day21_Rep2 | 1,145 | 111,594 | 498,856 | 0.2% | 22.4% |
| 5_a_Day21_Rep3 | 3,213 | 112,440 | 498,854 | 0.6% | 22.5% |

### PAM Sequence Interpretation

The PAM-protected sequence contains synonymous mutations that preserve BAP1 function, allowing cells to survive and become enriched as disruptive variants deplete from the population.

### REF Sequence Interpretation

The low starting percentage (~3%) indicates good HDR efficiency, and the steady depletion likely represents unedited cells that underwent Cas9 cutting followed by disruptive NHEJ-mediated indels.

---

## Summary: Key Takeaways

After completing these exercises, you should understand:

✅ **Adapter and primer trimming** - How to assess trimming efficiency and interpret cutadapt logs

✅ **Quality control** - How to use MultiQC to evaluate data quality across samples

✅ **Mapping statistics** - What good mapping rates look like and what the statistics mean

✅ **Selection patterns** - How to eyeball potential functional vs. disruptive variants from count changes

✅ **Biological interpretation** - How quantification data reveals variant function through negative selection

---

## Common Mistakes to Avoid

❌ **Expecting 100% mapping rates** - 40-60% is normal for SGE

❌ **Ignoring replicates** - Always check consistency across biological replicates

❌ **Misinterpreting REF depletion** - REF is unedited cells with NHEJ damage, not pure functional wild-type

❌ **Over-interpreting Day 4 data** - Selection takes time; focus on trends, not single timepoints

❌ **Forgetting sequencing depth** - Low-count variants may be noise; set minimum thresholds

❌ **Not recording tool versions** - Essential for reproducibility

---

**Congratulations on completing the tutorial!** These skills form the foundation for all SGE data analysis.

---

# Bonus Exercise Answers: Quantifying BAP1 Exon 7

---

## Task 1: Examine the Exon 7 Library

### Question B1.1: How many variants are in the exon 7 library?

**Answer:** 1,432 variants (after subtracting 1 for the header)

**Command used:**
```
wc -l data/exon7/valiant_library_7a.csv
# Result: 1433 lines total
# 1433 - 1 (header) = 1432 variants
```

**Comparison to Exon 5:** Exon 7 has **more variants** than exon 5 (1,432 vs 1,278).

---

## Task 2: Create the Samplesheet

**Answer:**

Create `data/exon7/samplesheet_7a.csv` with the following content:

```
sample,fastq_1,fastq_2
7_a_Day4_Rep1,data/exon7/fastq/7_a_Day4_Rep1_downsampled.fastq.gz,
7_a_Day4_Rep2,data/exon7/fastq/7_a_Day4_Rep2_downsampled.fastq.gz,
7_a_Day4_Rep3,data/exon7/fastq/7_a_Day4_Rep3_downsampled.fastq.gz,
```

**Key points:**
- ✓ Three samples (biological replicates)
- ✓ Naming follows convention: `[exon]_[library]_[timepoint]_[replicate]`
- ✓ Absolute paths using `${PWD}`
- ✓ Empty `fastq_2` column (single-end sequencing)

---

## Task 3: Determine Trimming Parameters

### Question B3.1: Why is the adapter the same for exon 7 as for exon 5?

Illumina adapters are sequencing infrastructure, not experiment-specific. All samples sequenced on the same Illumina platform use identical adapter sequences.

**For exon 7:** Use the same adapter as exon 5:
```
AGATCGGAAGAGCGGTTCAGCAGGAATGCCG
```

---

## Task 4: Determine Read Modification Parameters

### Question B4.1: Are append sequences the same or different from exon 5?

**Answer:** **SAME**

**Append sequences for exon 7:**
```
append_start: AATGATACGGCGACCACCGA
append_end:   TCGTATGCCGTCTTCTGCTTG
```

**Explanation:** These are **Illumina adapter remnants** from oligonucleotide synthesis, not biology-specific. All libraries synthesized using the same method will have identical append sequences.

**What stays constant across libraries:**
- ✓ Illumina sequencing adapters (technology)
- ✓ Append sequences (synthesis method)
- ✓ Quality score for appended bases (`?` = Q30)

**What changes between libraries:**
- ✗ Primers (specific to genomic region)
- ✗ Variant sequences (specific to target region)

---

## Task 5: Create the Configuration File

**Answer:**

Create `data/exon7/quants_7a.json` with:

```
{
  "max_cpus": 4,
  "single_end": true,
  "input_type": "fastq",
  "raw_sequencing_qc": true,
  "adapter_trimming": "cutadapt",
  "adapter_trimming_qc": true,
  "adapter_cutadapt_options": "-a AGATCGGAAGAGCGGTTCAGCAGGAATGCCG",
  "primer_trimming": "cutadapt",
  "primer_trimming_qc": true,
  "primer_cutadapt_options": "-a GCTGTGGGAGCTGATGTGGGG...GAGCTGGGGCTCAGGGCCCTCTGGTATGT",
  "read_modification": true,
  "append_start": "AATGATACGGCGACCACCGA",
  "append_end": "TCGTATGCCGTCTTCTGCTTG",
  "append_quality": "?",
  "transform_library": true,
  "pyquest_library_converter_options": "-N 1 -S 22",
  "quantification": "pyquest"
}
```

**Summary of parameters:**

| Parameter | Value | Why? |
|:----------|:------|:-----|
| Illumina adapter | `AGATCGGAAGAGCGGTTCAGCAGGAATGCCG` | Same platform as exon 5 |
| Forward primer | `GCTGTGGGAGCTGATGTGGGG` | Exon 7-specific |
| Reverse primer | `GAGCTGGGGCTCAGGGCCCTCTGGTATGT` | Exon 7-specific |
| Append start | `AATGATACGGCGACCACCGA` | Same synthesis method |
| Append end | `TCGTATGCCGTCTTCTGCTTG` | Same synthesis method |
| mseq column | 22 | Standard VaLiAnT format |

---

## Task 6: Run QUANTS

Now run QUANTS on the exon 7 data!

**Example command from exon5:**

```
nextflow run /home/manager/QUANTS/main.nf \
  -c ${PWD}/data/exon5/nextflow.config \
  -params-file ${PWD}/data/exon5/quants_5a.json \
  --input ${PWD}/data/exon5/samplesheet_5a.csv \
  --oligo_library ${PWD}/data/exon5/valiant_library_5a.csv \
  --outdir ${PWD}/results_5a
```
Update this command for exon7:

```
nextflow run /home/manager/QUANTS/main.nf \
  -c ${PWD}/data/exon7/nextflow.config \
  -params-file ${PWD}/data/exon7/quants_7a.json \
  --input ${PWD}/data/exon7/samplesheet_7a.csv \
  --oligo_library ${PWD}/data/exon7/valiant_library_7a.csv \
  --outdir ${PWD}/results_7a
```

---

## Summary: Key Learnings from Exon 7

✅ **Universal parameters** (same for all libraries):
- Illumina adapters
- Append sequences
- Basic QUANTS workflow

✅ **Library-specific parameters** (different per exon):
- Primers (match genomic region)
- Number of variants
- Target sequences

---

**Congratulations!** You've mastered SGE quantification independently! 🎉